[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traderbrasil/livro-dinheiro/blob/v2.0/notebooks/cap35_planilha_orcamento.ipynb)

# Capítulo 35: Seu plano financeiro em 6 etapas

Laboratório do livro *Dinheiro: Querer, poder e gastar* (2ª edição), de Flávio Lemos, CMT, CFP®.

Você abre a planilha de orçamento do livro (Apêndice A), lê os 12 meses e gera o resumo do ano em gráficos. Funciona com o exemplo de Paula e Marcos ou com a sua planilha preenchida.

**Como usar:** clique em cada célula e aperte o botão ▶ (ou Shift + Enter), de cima para baixo. Na primeira vez, o Colab pede para você entrar com uma conta do Google. Os campos marcados com `# MUDE AQUI` podem ser alterados com os seus números.

*Conteúdo educacional. Não é recomendação de investimento. Resultados passados não garantem resultados futuros. Os dados são baixados no dia em que você roda o notebook; por isso, os seus números podem ser diferentes dos do livro, que foram conferidos em outubro de 2026.*

In [ ]:
%pip install -q openpyxl
# Funções de apoio (não precisa mexer)
import io, time, math, datetime as dt
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES = {"azul": "#1F3864", "verde": "#2E7D32", "laranja": "#E07B00", "vermelho": "#C62828",
         "cinza": "#757575", "azul_claro": "#5B9BD5", "roxo": "#6A1B9A"}

def brl(x, casas=2):
    """Formata um número como R$ 1.234,56."""
    s = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "R$ " + s

def pct(x, casas=2):
    return f"{x*100:,.{casas}f}%".replace(",", "X").replace(".", ",").replace("X", ".")

def eixo_reais(ax):
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: brl(v, 0)))

def sgs(codigo, inicio="01/01/2000", fim=None):
    """Baixa uma série do Sistema Gerenciador de Séries do Banco Central (SGS).
    Séries diárias são baixadas em blocos de 10 anos, o limite da API."""
    fim = fim or dt.date.today().strftime("%d/%m/%Y")
    d0 = dt.datetime.strptime(inicio, "%d/%m/%Y").date()
    d1 = dt.datetime.strptime(fim, "%d/%m/%Y").date()
    partes = []
    while d0 <= d1:
        d_fim = min(d1, d0.replace(year=d0.year + 9, month=12, day=31))
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
               f"?formato=json&dataInicial={d0:%d/%m/%Y}&dataFinal={d_fim:%d/%m/%Y}")
        for tentativa in range(4):
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                dados = r.json()
                break
            except Exception:
                time.sleep(3 * (tentativa + 1))
        else:
            raise RuntimeError(f"Não consegui baixar a série {codigo}. Tente de novo em alguns minutos.")
        if dados:
            partes.append(pd.DataFrame(dados))
        d0 = d_fim + dt.timedelta(days=1)
    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], dayfirst=True)
    df["valor"] = pd.to_numeric(df["valor"])
    return df.set_index("data")["valor"].sort_index()


## 1. Escolha a planilha
Por padrão, o notebook usa o exemplo do livro. Para usar a sua, mude `USAR_MINHA` para `True` e rode a célula: o Colab vai pedir o arquivo.

In [ ]:
USAR_MINHA = False   # MUDE AQUI
URL_EXEMPLO = 'https://raw.githubusercontent.com/traderbrasil/livro-dinheiro/v2.0/planilhas/planilha-orcamento-dinheiro-exemplo.xlsx'
if USAR_MINHA:
    from google.colab import files
    up = files.upload(); ARQ = io.BytesIO(list(up.values())[0])
else:
    ARQ = io.BytesIO(requests.get(URL_EXEMPLO, timeout=60).content)
from openpyxl import load_workbook
wb = load_workbook(ARQ, data_only=False)
print('Abas:', ', '.join(wb.sheetnames))

## 2. Ler os meses
O notebook procura, em cada aba de mês, as linhas de renda, gastos e plano pela descrição da coluna A e soma os valores realizados da coluna C.

In [ ]:
MESES = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
GRUPOS = {'ESSENCIAIS': 'Essenciais', 'COMPROMISSOS': 'Compromissos', 'FLEXÍVEIS': 'Flexíveis', 'DESTINO DA MARGEM': 'Plano'}
def ler_mes(ws):
    tot = {'Renda': 0.0, 'Essenciais': 0.0, 'Compromissos': 0.0, 'Flexíveis': 0.0, 'Plano': 0.0, 'Irregulares pagas': 0.0}
    grupo = None
    for row in ws.iter_rows(min_row=4, max_col=3):
        a, c = row[0].value, row[2].value
        if isinstance(a, str) and not a.startswith('=') and a.split(' (')[0].isupper():
            grupo = next((v for k, v in GRUPOS.items() if a.startswith(k)), 'Renda' if a.startswith('RENDA') else None)
            continue
        if not isinstance(c, (int, float)):
            continue
        if isinstance(a, str) and a.startswith('Irregulares pagas'):
            tot['Irregulares pagas'] += c; continue
        if grupo:
            tot[grupo] += c
    return tot
res = pd.DataFrame({m: ler_mes(wb[m]) for m in MESES if m in wb.sheetnames}).T
res = res[res['Renda'] > 0]
res['Gastos'] = res['Essenciais'] + res['Compromissos'] + res['Flexíveis'] + res['Irregulares pagas']
res['Taxa de poupança'] = res['Plano'] / res['Renda']
res.style.format({c: (lambda v: brl(v)) for c in res.columns if c != 'Taxa de poupança'} | {'Taxa de poupança': lambda v: pct(v,1)})

*Observação: as fórmulas da planilha não são recalculadas aqui; o notebook soma os valores digitados nas linhas de cada grupo.*

## 3. O ano em gráficos

In [ ]:
fig, ax = plt.subplots()
x = np.arange(len(res)); w = 0.28
ax.bar(x - w, res['Renda'], w, label='Renda', color=CORES['azul'])
ax.bar(x, res['Gastos'], w, label='Gastos', color=CORES['laranja'])
ax.bar(x + w, res['Plano'], w, label='Plano (dívidas, reserva, metas)', color=CORES['verde'])
ax.set_xticks(x, res.index); eixo_reais(ax); ax.legend(); ax.set_title('Renda, gastos e plano por mês'); plt.show()
media = res[['Essenciais','Compromissos','Flexíveis','Plano']].mean()
media.plot.pie(autopct='%1.0f%%', colors=[CORES['azul'], CORES['azul_claro'], CORES['laranja'], CORES['verde']], title='Para onde vai a renda (média dos meses)', ylabel=''); plt.show()

Baixe a planilha em branco ou o exemplo na página do [Apêndice A](https://www.traderbrasil.com/livro-dinheiro/planilha/) e siga o roteiro das 6 etapas em [planejamento financeiro](https://www.traderbrasil.com/blog/planejamento-financeiro-como-organizar-seu-dinheiro.php).

---
**Hub do livro:** https://www.traderbrasil.com/livro-dinheiro/ · **Capítulo 35 no hub:** https://www.traderbrasil.com/livro-dinheiro/cap35/

Achou um erro? Escreva para contato@traderbrasil.com.